# 02: Preprocessing
Parse 6 ATT&CK techniques, extract features, save train/test splits

In [ ]:
import pandas as pd, numpy as np, json, os, glob
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
print('=== NOTEBOOK 02: PREPROCESSING ===')
RAW_DIR = '../data/raw'
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)
TECHNIQUE_MAP = {
    'T1566_Phishing': 'T1566',
    'T1110_BruteForce': 'T1110',
    'T1059_CommandShell': 'T1059',
    'T1055_ProcessInjection': 'T1055',
    'T1082_SystemInfo': 'T1082',
    'T1021_RemoteServices': 'T1021',
}
all_records = []
for folder_name, technique_id in TECHNIQUE_MAP.items():
    folder_path = os.path.join(RAW_DIR, folder_name)
    if not os.path.exists(folder_path):
        print(f'WARNING: {folder_path} not found')
        continue
    log_files = glob.glob(os.path.join(folder_path, '*.log'))
    print(f'Processing {technique_id}: {len(log_files)} file(s)')
    for log_file in log_files:
        count = 0
        with open(log_file, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try:
                    event = json.loads(line)
                    event['technique_id'] = technique_id
                    all_records.append(event)
                    count += 1
                except:
                    pass
        print(f'  {os.path.basename(log_file)}: {count} events')
print(f'\nTotal raw events: {len(all_records)}')
df = pd.DataFrame(all_records)
print(f'DataFrame shape: {df.shape}')
features = pd.DataFrame()
features['raw_length'] = df.apply(lambda row: len(str(row)), axis=1)
features['EventCode'] = pd.to_numeric(df.get('EventCode', pd.Series(np.nan, index=df.index)), errors='coerce').fillna(0)
features['source_type'] = df.get('sourcetype', df.get('SourceName', 'unknown'))
process_field = df.get('Image', df.get('ProcessName', df.get('NewProcessName', '')))
features['process_name'] = process_field.fillna('').astype(str)
features['process_name_length'] = features['process_name'].str.len()
features['has_process'] = (features['process_name'] != '').astype(int)
cmd_field = df.get('CommandLine', pd.Series('', index=df.index))
features['command_line'] = cmd_field.fillna('').astype(str)
features['command_line_length'] = features['command_line'].str.len()
features['has_command_line'] = (features['command_line'] != '').astype(int)
parent_field = df.get('ParentImage', df.get('ParentProcessName', ''))
features['has_parent_process'] = (parent_field.fillna('') != '').astype(int)
user_field = df.get('User', df.get('AccountName', df.get('SubjectUserName', '')))
features['user_present'] = (user_field.fillna('') != '').astype(int)
computer_field = df.get('Computer', df.get('ComputerName', 'unknown'))
features['computer'] = computer_field.fillna('unknown').astype(str)
time_field = df.get('_time', df.get('TimeCreated', ''))
time_parsed = pd.to_datetime(time_field, errors='coerce', utc=True)
features['hour_of_day'] = time_parsed.dt.hour.fillna(0)
features['day_of_week'] = time_parsed.dt.dayofweek.fillna(0)
features['log_type'] = df.get('Channel', df.get('LogName', 'unknown')).fillna('unknown')
target_field = df.get('TargetObject', df.get('TargetFilename', ''))
features['has_target'] = (target_field.fillna('') != '').astype(int)
features['technique_id'] = df['technique_id'].values
print(f'Feature matrix shape: {features.shape}')
le_technique = LabelEncoder()
features['technique_encoded'] = le_technique.fit_transform(features['technique_id'])
for col in ['source_type', 'process_name', 'command_line', 'computer', 'log_type']:
    le = LabelEncoder()
    features[col] = le.fit_transform(features[col].astype(str))
X = features.drop(['technique_id', 'technique_encoded'], axis=1)
y = features['technique_encoded']
print(f'\nClass distribution:\n{y.value_counts().sort_index()}')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'\nTrain: {X_train.shape}, Test: {X_test.shape}')
X_train.to_csv(f'{PROCESSED_DIR}/X_train.csv', index=False)
X_test.to_csv(f'{PROCESSED_DIR}/X_test.csv', index=False)
y_train.to_csv(f'{PROCESSED_DIR}/y_train.csv', index=False)
y_test.to_csv(f'{PROCESSED_DIR}/y_test.csv', index=False)
label_map_df = pd.DataFrame({'encoded': range(len(le_technique.classes_)), 'technique': le_technique.classes_})
label_map_df.to_csv(f'{PROCESSED_DIR}/label_map.csv', index=False)
pd.DataFrame({'feature': X.columns}).to_csv(f'{PROCESSED_DIR}/feature_names.csv', index=False)
print(f'\nSaved to {PROCESSED_DIR}/')
print(label_map_df.to_string(index=False))
print('\n=== PREPROCESSING COMPLETE ===')
